# Lila ASI v2 vs `llama-cpp-python` — Google Colab CPU Benchmark

This notebook benchmarks **Lila ASI v2 (`liblila_asi.so`)** against **`llama-cpp-python`** on Google Colab CPU hardware (typically Intel Xeon with **AVX-512F / AVX-512BW / AVX-512VNNI** or AMD EPYC with **AVX2 / AVX-512**).

### Before running
1. Upload a tarball of your `lila` source (`source/engine` + `source/src/core/asi_backend.py`) **or** set `GIT_REPO_URL` below.
   - To create a minimal tarball locally in WSL:
     ```bash
     cd /home/ticket/workspaces/lila
     tar -czf lila_asi_colab.tar.gz source/engine source/src/core/asi_backend.py
     ```
2. Make sure Runtime is set to **CPU** (or High-RAM CPU).

In [ ]:
#@title 1. Inspect Colab CPU & SIMD Capabilities
!lscpu | grep -E 'Model name|CPU\(s\)|Thread|Core|Socket|Flags' | head -n 10
import re
flags = set(re.findall(r'\S+', open('/proc/cpuinfo').read().split('flags')[1].split('\n')[0]))
for f in ['avx2', 'fma', 'f16c', 'avx_vnni', 'avx512f', 'avx512bw', 'avx512vl', 'avx512_vnni']:
    print(f"{f:14s}: {'YES' if f in flags else 'no'}")

In [ ]:
#@title 2. Load Lila ASI Source (Git clone OR upload lila_asi_colab.tar.gz)
import os
GIT_REPO_URL = ""  #@param {type:"string"}
GIT_BRANCH = "feature/voice-input-portable"  #@param {type:"string"}

os.makedirs("/content/lila/source", exist_ok=True)
%cd /content/lila

if GIT_REPO_URL:
    !rm -rf /content/lila_repo
    if GIT_BRANCH:
        !git clone --depth 1 --branch "{GIT_BRANCH}" "{GIT_REPO_URL}" /content/lila_repo
    else:
        !git clone --depth 1 "{GIT_REPO_URL}" /content/lila_repo
    !echo "Git HEAD: $(git -C /content/lila_repo rev-parse HEAD)"
    !git -C /content/lila_repo status --short
    if os.path.isdir("/content/lila_repo/source/engine"):
        !cp -r /content/lila_repo/source/. /content/lila/source/
    else:
        !cp -r /content/lila_repo/. /content/lila/source/
elif os.path.exists("/content/lila_asi_colab.tar.gz"):
    !tar -xzf /content/lila_asi_colab.tar.gz -C /content/lila
else:
    from google.colab import files
    print("Upload lila_asi_colab.tar.gz (created via: tar -czf lila_asi_colab.tar.gz source/engine source/src/core/asi_backend.py)")
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    !tar -xzf "{fname}" -C /content/lila

!ls -la /content/lila/source/engine

In [ ]:
#@title 3. Build liblila_asi.so & Run Scalar-vs-SIMD Kernel Parity Check
%cd /content/lila/source/engine
!command -v nasm >/dev/null || (apt-get update -qq && apt-get install -y -qq nasm)
!make clean && make lila-engine lila-asi liblila-asi test-v2
!gcc -O3 -march=native -Wall -Wextra -std=c11 -pthread -fopenmp -I runtime/ tests/test_quant_kernels.c -o /tmp/test_quant_kernels -lm -lpthread -fopenmp && /tmp/test_quant_kernels
!ls -lh asi/liblila_asi.so

In [ ]:
#@title 4. Install Python dependencies (llama-cpp-python, huggingface_hub)
!pip install -q "huggingface_hub>=0.25.0,<2.0"
!CMAKE_ARGS="-DGGML_NATIVE=ON" pip install -q llama-cpp-python --no-binary llama-cpp-python || pip install -q llama-cpp-python

In [ ]:
#@title 5. Download Qwen3.5-0.8B GGUF & Convert to .asi v2 Cartridge
from huggingface_hub import hf_hub_download
import os

HF_REPO = "mradermacher/Qwen3.5-0.8B-heretic-i1-GGUF"  #@param {type:"string"}
HF_FILE = "Qwen3.5-0.8B-heretic.i1-Q4_K_M.gguf"  #@param {type:"string"}

gguf_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, local_dir="/content/models")
asi_path = "/content/qwen35_0.8b.asi"
print("Downloaded:", gguf_path)

!python3 /content/lila/source/engine/asi/gguf_to_asi_v2.py "{gguf_path}" "{asi_path}"
!ls -lh "{asi_path}"

import hashlib
def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for _label, _p in [("gguf_path", gguf_path), ("asi_path", asi_path)]:
    _digest = _sha256_file(_p)
    print(f"{_label} sha256: {_digest[:16]}  full={_digest}  ({_p})")

In [ ]:
#@title 6. Gate: asi_repeat Determinism & Correctness (Fresh Process, no llama_cpp)
import subprocess, sys

_GATE_SCRIPT = """
import sys
sys.path.insert(0, '/content/lila/source/src')
assert 'llama_cpp' not in sys.modules, 'llama_cpp must not be imported'
from core.asi_backend import AsiLlamaAdapter

asi_path = sys.argv[1]
n_threads = int(sys.argv[2])
n_runs = int(sys.argv[3])
messages = [
    {'role': 'system', 'content': 'You are a concise assistant.'},
    {'role': 'user', 'content': 'Count from 1 to 10 separated by commas, starting with 1.'},
]
outputs = []
for _ in range(n_runs):
    llm = AsiLlamaAdapter(asi_path, n_ctx=512, n_threads=n_threads, verbose=False)
    try:
        resp = llm.create_chat_completion(
            messages=messages, max_tokens=32, temperature=0.0, top_p=1.0, top_k=1
        )
        outputs.append(resp['choices'][0]['message']['content'].strip())
    finally:
        llm.close()

det = (len(set(outputs)) == 1)
ok = ('1, 2, 3, 4' in outputs[0])
print(f'threads={n_threads} N={n_runs}:', 'DETERMINISTIC' if det else 'NON-DETERMINISTIC', '|', 'CORRECT-LOOKING' if ok else 'CORRUPTED')
print('sample:', repr(outputs[0][:120]))
if not (det and ok):
    sys.exit(1)
"""

for _t in (1, 2):
    _proc = subprocess.run(
        [sys.executable, "-c", _GATE_SCRIPT, asi_path, str(_t), "3"],
        capture_output=True,
        text=True,
    )
    print(_proc.stdout, end="")
    if _proc.stderr:
        print(_proc.stderr, end="")
    if (
        _proc.returncode != 0
        or "DETERMINISTIC" not in _proc.stdout
        or "CORRECT-LOOKING" not in _proc.stdout
    ):
        raise RuntimeError(f"asi_repeat gate failed for threads={_t} (rc={_proc.returncode})")
print("asi_repeat gate PASSED for threads=1 and threads=2 (N=3)")

In [ ]:
#@title 7. Hardware Passport, Correctness Gate (N_GEN=64), & Strict Benchmark (N=10, N_GEN=32)
import os, platform, re, statistics, subprocess, sys, time
sys.path.insert(0, "/content/lila/source/src")

N_THREADS = 0  #@param {type:"integer"}
N_GEN = 32  #@param {type:"integer"}
N_RUNS = 10  #@param {type:"integer"}
CORR_N_GEN = 64  #@param {type:"integer"}
THREADS = N_THREADS if N_THREADS > 0 else (os.cpu_count() or 2)
TESTS_DIR = "/content/lila/source/engine/tests"
LIB_PATH = "/content/lila/source/engine/asi/liblila_asi.so"
CORRECTNESS_JSON = "/content/correctness_colab.json"
BENCH_JSON = "/content/bench_colab.json"

def _print_hardware_passport(threads_used):
    cpu_model = platform.processor() or "unknown"
    vendor_id = "unknown"
    flags = set()
    phys_cores = set()
    if os.path.exists("/proc/cpuinfo"):
        cpuinfo = open("/proc/cpuinfo", "r", encoding="utf-8", errors="ignore").read()
        m_model = re.search(r"^model name\s*:\s*(.+)$", cpuinfo, re.MULTILINE)
        if m_model:
            cpu_model = m_model.group(1).strip()
        m_vendor = re.search(r"^vendor_id\s*:\s*(.+)$", cpuinfo, re.MULTILINE)
        if m_vendor:
            vendor_id = m_vendor.group(1).strip()
        m_flags = re.search(r"^(?:flags|Features)\s*:\s*(.+)$", cpuinfo, re.MULTILINE)
        if m_flags:
            flags = set(m_flags.group(1).split())
        cur_pkg, cur_core = "0", None
        for line in cpuinfo.splitlines():
            if line.startswith("physical id"):
                cur_pkg = line.split(":", 1)[1].strip()
            elif line.startswith("core id"):
                cur_core = line.split(":", 1)[1].strip()
                phys_cores.add((cur_pkg, cur_core))
    logical_cpus = os.cpu_count() or 1
    core_count = len(phys_cores) if phys_cores else logical_cpus
    ram_total_gib, ram_avail_gib = 0.0, 0.0
    if os.path.exists("/proc/meminfo"):
        meminfo = open("/proc/meminfo", "r", encoding="utf-8", errors="ignore").read()
        mt = re.search(r"^MemTotal:\s+(\d+)\s+kB", meminfo, re.MULTILINE)
        ma = re.search(r"^MemAvailable:\s+(\d+)\s+kB", meminfo, re.MULTILINE)
        if mt:
            ram_total_gib = int(mt.group(1)) / (1024.0 * 1024.0)
        if ma:
            ram_avail_gib = int(ma.group(1)) / (1024.0 * 1024.0)
    simd_keys = ["avx2", "fma", "f16c", "avx_vnni", "avx512f", "avx512bw", "avx512vl", "avx512_vnni"]
    simd_present = [k for k in simd_keys if k in flags]
    if "avx512f" in flags and "avx512_vnni" in flags:
        isa_tier = "AVX-512 + VNNI"
    elif "avx512f" in flags:
        isa_tier = "AVX-512"
    elif "avx2" in flags:
        isa_tier = "AVX2 + FMA" if "fma" in flags else "AVX2"
    else:
        isa_tier = platform.machine()
    print("=== CPU & Architecture Hardware Passport ===")
    print(f"Arch / Vendor : {platform.machine()} ({vendor_id})")
    print(f"CPU Model     : {cpu_model}")
    print(f"Cores/Threads : {core_count} physical cores / {logical_cpus} logical CPUs (bench threads={threads_used})")
    print(f"ISA Tier      : {isa_tier}")
    print(f"SIMD Flags    : {', '.join(simd_present) if simd_present else 'none'}")
    print(f"System RAM    : {ram_total_gib:.2f} GiB total ({ram_avail_gib:.2f} GiB available)")

_print_hardware_passport(THREADS)

strict_bar_script = os.path.join(TESTS_DIR, "bench_strict_bar.py")
correctness_script = os.path.join(TESTS_DIR, "test_correctness_gate.py")
qualify_script = os.path.join(TESTS_DIR, "qualify_asi_backend.py")

if os.path.exists(correctness_script):
    print(f"\n=== Running test_correctness_gate.py (n_gen={CORR_N_GEN}, threads={THREADS}) ===")
    _corr_env = os.environ.copy()
    _corr_env["LILA_ASI_PATH"] = asi_path
    _corr_env["LILA_GGUF_PATH"] = gguf_path
    _corr_env["LILA_N_THREADS"] = str(THREADS)
    _r = subprocess.run(
        [
            sys.executable, correctness_script,
            "--asi", asi_path,
            "--gguf", gguf_path,
            "--n-threads", str(THREADS),
            "--n-gen", str(CORR_N_GEN),
            "--json-out", CORRECTNESS_JSON,
        ],
        env=_corr_env,
        check=False,
    )
    if _r.returncode != 0:
        raise RuntimeError(f"test_correctness_gate.py failed (rc={_r.returncode})")
    subprocess.run([sys.executable, "-m", "json.tool", CORRECTNESS_JSON], stdout=subprocess.DEVNULL, check=True)
    print(f"Correctness JSON validated & saved: {CORRECTNESS_JSON}")

if os.path.exists(strict_bar_script):
    print(f"\n=== Running bench_strict_bar.py (N={N_RUNS}, n_gen={N_GEN}, threads={THREADS}, identical token IDs) ===")
    _proc = subprocess.run(
        [
            sys.executable, strict_bar_script,
            "--n", str(N_RUNS),
            "--asi", asi_path,
            "--gguf", gguf_path,
            "--lib", LIB_PATH,
            "--n-threads", str(THREADS),
            "--n-gen", str(N_GEN),
            "--json-out", BENCH_JSON,
        ],
        capture_output=True,
        text=True,
    )
    print(_proc.stdout)
    if _proc.stderr:
        print(_proc.stderr)
    if _proc.returncode != 0:
        raise RuntimeError(f"bench_strict_bar.py failed (rc={_proc.returncode})")
    subprocess.run([sys.executable, "-m", "json.tool", BENCH_JSON], stdout=subprocess.DEVNULL, check=True)
    print(f"Benchmark JSON validated & saved: {BENCH_JSON}")
    print(f"Export paths for paper: correctness={CORRECTNESS_JSON} | bench={BENCH_JSON}")
else:
    from core.asi_backend import AsiLlamaAdapter
    from llama_cpp import Llama

    MESSAGES = [
        {"role": "system", "content": "You are Lila, a concise local AI assistant."},
        {"role": "user", "content": "Give me a 1-sentence greeting and explain what 2+2 equals."},
    ]

    def bench_once(loader):
        t0 = time.monotonic()
        llm = loader()
        load_s = time.monotonic() - t0
        res = {"load": load_s}
        try:
            for turn in (1, 2):
                t1 = time.monotonic()
                ttft = None
                parts = []
                for ev in llm.create_chat_completion(
                    messages=MESSAGES, max_tokens=N_GEN, temperature=0.0, top_p=1.0, top_k=1, stream=True
                ):
                    delta = (ev.get("choices") or [{}])[0].get("delta", {}).get("content")
                    if delta:
                        if ttft is None:
                            ttft = time.monotonic() - t1
                        parts.append(delta)
                total_s = time.monotonic() - t1
                gen_s = max(1e-6, total_s - (ttft or total_s))
                tok_s = max(0, len(parts) - 1) / gen_s if parts else 0.0
                res[f"t{turn}_ttft"] = ttft or total_s
                res[f"t{turn}_dec"] = tok_s
                res[f"t{turn}_text"] = "".join(parts).strip()[:120]
        finally:
            if hasattr(llm, "close"):
                llm.close()
        return res

    asi_runs = [bench_once(lambda: AsiLlamaAdapter(asi_path, n_ctx=512, n_threads=THREADS, verbose=False)) for _ in range(N_RUNS)]
    llc_runs = [bench_once(lambda: Llama(model_path=gguf_path, n_ctx=512, n_threads=THREADS, verbose=False)) for _ in range(N_RUNS)]

    print(f"\n=== Controlled Medians (N={N_RUNS}, threads={THREADS}) ===")
    for k in ("load", "t1_ttft", "t1_dec", "t2_ttft", "t2_dec"):
        av = [r[k] for r in asi_runs]
        lv = [r[k] for r in llc_runs]
        unit = "tok/s" if "dec" in k else "s"
        print(f"{k:8s}: ASI med={statistics.median(av):6.2f} [{min(av):.2f}..{max(av):.2f}] {unit} | llama.cpp med={statistics.median(lv):6.2f} [{min(lv):.2f}..{max(lv):.2f}] {unit}")
    print("ASI sample:", asi_runs[0]["t1_text"])
    print("LLC sample:", llc_runs[0]["t1_text"])

if os.path.exists(qualify_script):
    print("\n=== Running qualify_asi_backend.py checks ===")
    _qual_code = (
        f"import os, sys; sys.path.insert(0, {TESTS_DIR!r}); "
        f"import qualify_asi_backend as q; "
        f"q.GGUF_PATH = {gguf_path!r}; q.ASI_PATH = {asi_path!r}; q.N_THREADS = {THREADS}; "
        f"q.test_1_correctness(); "
        f"q.test_2_multiturn_long_toolcall(); "
        f"q.test_3_cancel_isolation_simultaneous(); "
        f"q.test_4_memory_and_prefix_invalidation(); "
        f"(q.test_5_daemon_with_lila_backend_asi() if os.path.exists('/content/lila/source/src/core/daemon.py') else print('[5] Skipped daemon test (daemon.py not in tarball)')); "
        f"print('QUALIFICATION GATE PASSED')"
    )
    _qr = subprocess.run([sys.executable, "-c", _qual_code], check=False)
    if _qr.returncode != 0:
        raise RuntimeError(f"qualify_asi_backend.py failed (rc={_qr.returncode})")